# nf-core/sarek Test Run

Germline/somatic variant calling pipeline using Nextflow.

## Prerequisites

**Required for successful execution:**
- Docker or Singularity (for containers)
- OR properly configured conda environment
- Nextflow 24.04.4 or compatible version
- For AWS Batch: IAM roles, compute environment, job queue configured

**Best execution environments:**
- EC2 instance with Docker daemon running
- HPC cluster with Slurm + Singularity
- AWS Batch (for cloud-native execution)

## Pipeline Info

- **Pipeline**: nf-core/sarek
- **Version**: 3.4.0 (compatible with Nextflow 24.04.4)
- **Purpose**: Variant calling from DNA sequencing data
- **Test profile**: Uses small public test dataset (~5-10 min on AWS Batch)
- **Tools**: Includes FastQC, BWA, GATK, Strelka, and more

## Installation

In [ ]:
# Install Nextflow 24.04.4 (compatible with sarek 3.4.0)
# Run this once per environment
!conda install -c bioconda nextflow=24.04.4 -y

In [1]:
%%sh
# Verify installation
nextflow -version


      N E X T F L O W
      version 24.04.4 build 5917
      created 01-08-2024 07:05 UTC 
      cite doi:10.1038/nbt.3820
      http://nextflow.io



## Configuration

Choose ONE of the following configurations based on your environment.

### Option 1: Local Execution with Docker (Recommended for EC2)

In [ ]:
%%sh
mkdir -p ~/nextflow_config

cat > ~/nextflow_config/local_docker.config << 'EOF'
process {
    executor = 'local'
    cpus = 8
    memory = '30 GB'
}

docker {
    enabled = true
    runOptions = '-u $(id -u):$(id -g)'
}
EOF

echo "✅ Local Docker config created at ~/nextflow_config/local_docker.config"

### Option 2: AWS Batch Execution

In [ ]:
%env AWS_ACCESS_KEY_ID="enter here without quotes"
%env AWS_SECRET_ACCESS_KEY="enter here without quotes"
%env AWS_SESSION_TOKEN="enter here without quotes" 

In [ ]:
%%sh
mkdir -p ~/nextflow_config

cat > ~/nextflow_config/aws_batch.config << 'EOF'
process {
    executor = 'awsbatch'
    queue = 'YOUR-BATCH-QUEUE-NAME'  // UPDATE THIS
}

aws {
    region = 'us-east-1'  // UPDATE THIS
    batch {
        cliPath = '/usr/local/bin/aws'
        jobRole = 'arn:aws:iam::ACCOUNT:role/YOUR-ECS-TASK-ROLE'  // UPDATE THIS
    }
}
EOF

echo "✅ AWS Batch config created at ~/nextflow_config/aws_batch.config"
echo "⚠️  IMPORTANT: Edit the file to add your queue name, region, and IAM role ARN"

### Option 3: Local with Singularity (for HPC clusters)

In [ ]:
%%sh
mkdir -p ~/nextflow_config

cat > ~/nextflow_config/local_singularity.config << 'EOF'
process {
    executor = 'local'
    cpus = 8
    memory = '30 GB'
}

singularity {
    enabled = true
    autoMounts = true
}
EOF

echo "✅ Singularity config created at ~/nextflow_config/local_singularity.config"

## Run Pipeline

### Test Run (Local with Docker)

In [ ]:
%%sh
# Create working directory
mkdir -p ~/sarek_test_run
cd ~/sarek_test_run

# Run sarek test profile with local Docker
nextflow run nf-core/sarek -r 3.4.0 \
  -profile test,docker \
  -c ~/nextflow_config/local_docker.config \
  -work-dir $PWD/work \
  --outdir $PWD/results \
  -with-report $PWD/report.html \
  -with-timeline $PWD/timeline.html \
  -with-trace $PWD/trace.txt

### Test Run (AWS Batch)

In [ ]:
%%sh
# Run sarek test profile on AWS Batch
# Results stored in S3

S3_BUCKET="s3://YOUR-BUCKET-NAME"  # UPDATE THIS

nextflow run nf-core/sarek -r 3.4.0 \
  -profile test \
  -c ~/nextflow_config/aws_batch.config \
  -work-dir ${S3_BUCKET}/sarek_work \
  --outdir ${S3_BUCKET}/sarek_results \
  -with-report report.html \
  -with-timeline timeline.html \
  -with-trace trace.txt

### Resume Failed Run

If the pipeline fails, you can resume from where it stopped:

In [ ]:
%%sh
cd ~/sarek_test_run

# Add -resume flag to continue from last successful step
nextflow run nf-core/sarek -r 3.4.0 \
  -profile test,docker \
  -c ~/nextflow_config/local_docker.config \
  -work-dir $PWD/work \
  --outdir $PWD/results \
  -resume

## Check Results

In [ ]:
%%sh
# List output files
echo "=== Output Directory Structure ==="
tree -L 2 ~/sarek_test_run/results || ls -lR ~/sarek_test_run/results

echo ""
echo "=== Execution Reports ==="
ls -lh ~/sarek_test_run/*.html ~/sarek_test_run/*.txt

## Understanding the Output

**Key output directories:**
- `preprocessing/`: Aligned BAM files, QC reports
- `variant_calling/`: VCF files with called variants
- `reports/`: MultiQC summary reports
- `pipeline_info/`: Execution metadata

**Execution reports:**
- `report.html`: Resource usage, task completion status
- `timeline.html`: Visual timeline of task execution
- `trace.txt`: Detailed metrics per task (CPU, memory, duration)

## Troubleshooting

### Common Issues

**1. Docker daemon not running**
```bash
# Check Docker status
docker ps

# Start Docker (requires sudo)
sudo systemctl start docker
```

**2. S3 Access Denied errors (AWS Batch)**
- Verify IAM role has S3 read/write permissions
- Check bucket name is correct
- Ensure ECS task role is attached to Batch jobs

**3. Out of memory errors**
- Increase memory in config: `memory = '60 GB'`
- For AWS Batch: increase instance type in compute environment

**4. Container pull failures**
- Check internet connectivity
- For private registries: configure Docker/Singularity credentials

**5. Version compatibility**
```bash
# Check versions
nextflow -version
docker --version

# Use specific sarek version
nextflow run nf-core/sarek -r 3.4.0  # Pin version
```

### View Logs

```bash
# Nextflow log
cat .nextflow.log

# Failed task details
cd work/<hash>/  # Get hash from error message
cat .command.log
cat .command.err
```

## Next Steps

### Run with Your Own Data

1. **Prepare sample sheet** (CSV format):
```csv
patient,sample,lane,fastq_1,fastq_2
patient1,sample1,lane1,/path/to/R1.fastq.gz,/path/to/R2.fastq.gz
```

2. **Run pipeline**:
```bash
nextflow run nf-core/sarek -r 3.4.0 \
  --input samplesheet.csv \
  --genome GRCh38 \
  --tools strelka,mutect2 \
  -profile docker \
  -c ~/nextflow_config/local_docker.config \
  --outdir results/
```

### Additional Resources

- [Sarek documentation](https://nf-co.re/sarek/3.4.0)
- [Nextflow documentation](https://www.nextflow.io/docs/latest/)
- [nf-core best practices](https://nf-co.re/docs/usage/getting_started)
- [AWS Batch configuration](https://www.nextflow.io/docs/latest/awscloud.html#aws-batch)